# 출시 전 LLM 체크리스트용 데이터 생성

이 코드은 보고서용 그래프를 만드는 코드이 아니다.  
`03_run_llm_allgames_analysi.ipynb`에서 생성한 LLM 분석 결과를 읽고,  
나중에 LLM이 출시 전 체크리스트를 만들 때 사용할 CSV 파일을 생성한다.

사용자 조건 입력과 LLM 호출은 이 코드에서 하지 않는다.

# 1. 목적

최종 목표는 개발자가 입력한 장르, 가격대, Steam 태그, 플레이 방식에 따라  
출시 전 점검 항목을 LLM이 생성할 수 있도록 근거 데이터를 미리 정리하는 것이다.

다만 체크리스트의 **우선순위는 LLM이 새로 판단하지 않도록 한다.**  
이 코드에서 조건별 반복 이슈를 먼저 계산하고, **사전에 정한 규칙 기반 기준으로 상·중·하 우선순위를 확정한다.**

우선순위 판단에는 다음 내용을 사용한다.

| 기준 | 의미 | 우선순위 반영 방식 |
|---|---|---|
| 여러 게임에서 반복되는지 | 특정 리뷰 몇 개가 아니라, 여러 게임에서 공통적으로 나타난 문제인지 확인 | 핵심 기준 |
| 조건 안에서 자주 나타나는지 | 특정 장르·가격대·Steam 태그·플레이 방식 안에서 자주 보이는 이슈인지 확인 | 핵심 기준 |
| Steam 비추천 맥락과 연결되는지 | 해당 이슈가 Steam 비추천 리뷰가 있는 게임에서도 반복되는지 확인 | 핵심 기준 |
| High urgency가 반복되는지 | 이전 LLM 리뷰 분석에서 강한 문제로 분류된 사례가 여러 게임에서 나타나는지 확인 | **우선순위 계산에는 사용하지 않고 보조 설명으로만 사용** |

중요한 점은 `High urgency`가 이전 LLM 분석 결과에서 나온 값이라는 것이다.  
따라서 `High urgency`만으로 우선순위를 올리지 않으며, 이번 코드의 `priority_level` 계산식에도 직접 넣지 않는다.

이 코드에서는 다음 4개 CSV를 생성한다.

| 파일명 | 역할 |
|---|---|
| `prelaunch_game_base.csv` | 게임 단위 기본 정보 |
| `prelaunch_issue_repeat_summary.csv` | 전체 이슈 반복성 요약 |
| `prelaunch_condition_issue_summary.csv` | 조건별 이슈 요약 및 규칙 기반 우선순위 |
| `prelaunch_checklist_evidence_base.csv` | LLM 입력용 근거 문장 데이터 |


# 2. 기본 설정

In [1]:
# ============================================================
# 기본 라이브러리
# ============================================================
import ast
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

In [2]:
# ============================================================
# 프로젝트 경로 설정
# ============================================================
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정한다.
from pathlib import Path

ROOT = Path.cwd()

# Jupyter 실행 위치가 하위 폴더일 수 있으므로,
# data/preprocessed 폴더가 보일 때까지 상위 폴더를 탐색한다.
if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# ============================================================
# 출시 전 master 데이터 폴더
# ============================================================
PRELAUNCH_OUTPUT_DIR = ROOT / "data" / "outputs" / "prelaunch"
MASTER_DIR = PRELAUNCH_OUTPUT_DIR / "master"
MASTER_DIR.mkdir(parents=True, exist_ok=True)

# 하위 산출물 폴더
CHECKLIST_DATA_DIR = MASTER_DIR / "prelaunch_checklist_data"
TABLEAU_DIR = MASTER_DIR / "tableau_dashboard_csv"

CHECKLIST_DATA_DIR.mkdir(parents=True, exist_ok=True)
TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# LLM 실행 산출물 호출 경로
# ============================================================
RESULT_CSV_PATH = MASTER_DIR / "llm_review_analysis_result.csv"
ISSUE_TAG_FLAT_PATH = MASTER_DIR / "llm_issue_tags_flat.csv"
LLM_INPUT_PATH = MASTER_DIR / "llm_input_reviews.csv"

# 게임 메타데이터
GRADED_GAMES_PATH = ROOT / "data" / "preprocessed" / "steam_indie_games_graded.csv"

# ============================================================
# 체크리스트용 출력 파일
# ============================================================
GAME_BASE_PATH = CHECKLIST_DATA_DIR / "prelaunch_game_base.csv"
ISSUE_REPEAT_SUMMARY_PATH = CHECKLIST_DATA_DIR / "prelaunch_issue_repeat_summary.csv"
CONDITION_ISSUE_SUMMARY_PATH = CHECKLIST_DATA_DIR / "prelaunch_condition_issue_summary.csv"
CHECKLIST_EVIDENCE_BASE_PATH = CHECKLIST_DATA_DIR / "prelaunch_checklist_evidence_base.csv"

# ============================================================
# Tableau용 출력 파일
# ============================================================
TABLEAU_SOURCE_PATH = TABLEAU_DIR / "tableau_allgames_prelaunch_source.csv"

print("ROOT:", ROOT)
print("MASTER_DIR:", MASTER_DIR)
print("RESULT_CSV_PATH exists:", RESULT_CSV_PATH.exists())
print("ISSUE_TAG_FLAT_PATH exists:", ISSUE_TAG_FLAT_PATH.exists())
print("LLM_INPUT_PATH exists:", LLM_INPUT_PATH.exists())
print("GRADED_GAMES_PATH exists:", GRADED_GAMES_PATH.exists())

ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
MASTER_DIR: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master
RESULT_CSV_PATH exists: True
ISSUE_TAG_FLAT_PATH exists: True
LLM_INPUT_PATH exists: True
GRADED_GAMES_PATH exists: True


In [3]:
# ============================================================
# 분석 기준 설정
# ============================================================
# 튜터님 피드백 반영:
# 우선순위는 LLM이 새로 판단하지 않고, 사전에 정한 데이터 기준으로 계산한다.
#
# 핵심 기준:
# 1. 여러 게임에서 반복되는가
# 2. 조건 안에서 발생 비율이 높은가
# 3. Steam 비추천 리뷰가 있는 게임에서도 반복되는가
#
# 보조 지표:
# - high_urgency_game_count는 이전 LLM 리뷰 분석에서 나온 값이다.
# - 따라서 priority_level 계산에는 직접 사용하지 않는다.
# - 단, 최종 근거 문장과 대시보드에서 "참고 지표"로만 남긴다.

GAME_ID_COL = "appid"
GAME_NAME_COL = "game_name"
REVIEW_ID_COL = "recommendationid"
ISSUE_COL = "issue_name_kor"

# ------------------------------------------------------------
# 우선순위 기준
# ------------------------------------------------------------
# 04번 출시 후 분석과 기준 철학을 통일한다.
#
# 1단계: 절대 기준으로 priority_candidate를 만든다.
# 2단계: 조건별 상대 보정으로 최종 priority_level을 확정한다.
#
# 단, 03-1은 출시 전 분석이므로 리뷰 수가 아니라 게임 수 기준을 사용한다.
# High urgency는 LLM 보조 지표이므로 우선순위 계산에는 직접 사용하지 않는다.

HIGH_PRIORITY_MIN_ISSUE_GAMES = 5
HIGH_PRIORITY_MIN_STEAM_NEGATIVE_GAMES = 3
HIGH_PRIORITY_MIN_RATIO = 30.0
HIGH_PRIORITY_MIN_BASE_GAMES = 5

MID_PRIORITY_MIN_ISSUE_GAMES = 3
MID_PRIORITY_MIN_STEAM_NEGATIVE_GAMES = 2
MID_PRIORITY_MIN_RATIO = 15.0
MID_PRIORITY_MIN_BASE_GAMES = 3

# ------------------------------------------------------------
# 최종 '상' 개수 제한 기준
# ------------------------------------------------------------
# 04번 출시 후 분석과 동일하게,
# '상 후보 → 최종 상 제한 → 나머지는 중 조정' 구조를 사용한다.
#
# 조건별 최대 6개
# 조건별 개선 대상 이슈의 30% 이내

MAX_HIGH_ISSUES = 6
MAX_HIGH_RATE = 0.30

# 긍정 칭찬은 강화 요소이고, 기타는 원인이 명확하지 않으므로 최종 '상'에서 제외한다.
NO_HIGH_ISSUES = {
    "긍정 칭찬",
    "기타",
    "positive_praise",
    "other",
}

# LLM 입력용 근거표에서 조건별로 남길 최대 이슈 수
TOP_N_EVIDENCE_PER_CONDITION = 15

priority_order_map = {
    "상": 1,
    "중": 2,
    "하": 3,
}


# 3. 데이터 불러오기 및 기본 전처리

In [4]:
# ============================================================
# 데이터 불러오기
# ============================================================
review_df = pd.read_csv(RESULT_CSV_PATH)
tag_df = pd.read_csv(ISSUE_TAG_FLAT_PATH)
llm_input_df = pd.read_csv(LLM_INPUT_PATH)

In [5]:
# ============================================================
# 기본 컬럼 타입 정리
# ============================================================
# 세 파일을 recommendationid와 appid로 연결하기 위해 타입을 맞춘다.
# recommendationid는 숫자처럼 보여도 고유 ID이므로 문자열로 둔다.
# appid는 게임 식별용 숫자 키이므로 int로 맞춘다.

review_df[REVIEW_ID_COL] = review_df[REVIEW_ID_COL].astype(str)
tag_df[REVIEW_ID_COL] = tag_df[REVIEW_ID_COL].astype(str)
llm_input_df[REVIEW_ID_COL] = llm_input_df[REVIEW_ID_COL].astype(str)

review_df[GAME_ID_COL] = review_df[GAME_ID_COL].astype(int)
tag_df[GAME_ID_COL] = tag_df[GAME_ID_COL].astype(int)
llm_input_df[GAME_ID_COL] = llm_input_df[GAME_ID_COL].astype(int)

# 비교/필터링이 흔들리지 않도록 주요 범주형 컬럼은 소문자와 공백 제거 기준으로 통일한다.
review_df["llm_sentiment"] = review_df["llm_sentiment"].astype(str).str.strip().str.lower()
review_df["steam_label_text"] = review_df["steam_label_text"].astype(str).str.strip().str.lower()
review_df["llm_urgency_candidate"] = review_df["llm_urgency_candidate"].astype(str).str.strip().str.lower()

tag_df["llm_sentiment"] = tag_df["llm_sentiment"].astype(str).str.strip().str.lower()
tag_df["steam_label_text"] = tag_df["steam_label_text"].astype(str).str.strip().str.lower()
tag_df["llm_issue_sentiment"] = tag_df["llm_issue_sentiment"].astype(str).str.strip().str.lower()
tag_df["llm_urgency_candidate"] = tag_df["llm_urgency_candidate"].astype(str).str.strip().str.lower()
tag_df[ISSUE_COL] = tag_df[ISSUE_COL].astype(str).str.strip()

In [6]:
# ============================================================
# LLM 입력 파일의 메타데이터 결합
# ============================================================
# LLM 결과 파일은 리뷰 분석 결과 중심이라 가격대, 장르, Steam 태그, 카테고리 정보가 빠질 수 있다.
# 그래서 LLM 호출 당시 입력으로 사용한 llm_input_reviews.csv에서 메타데이터를 다시 붙인다.

meta_cols = [
    REVIEW_ID_COL,
    "price",
    "price_group",
    "genres_text",
    "categories_text",
    "top_steam_tags_text"
]

# recommendationid 기준으로 1개 리뷰당 1개의 메타데이터만 남긴다.
meta_df = llm_input_df[meta_cols].drop_duplicates(REVIEW_ID_COL)

add_cols = ["price", "price_group", "genres_text", "categories_text", "top_steam_tags_text"]

# 기존에 같은 컬럼이 있으면 중복 생성을 막기 위해 먼저 제거하고 다시 결합한다.
review_df = review_df.drop(columns=add_cols, errors="ignore")
review_df = review_df.merge(meta_df, on=REVIEW_ID_COL, how="left")

tag_df = tag_df.drop(columns=add_cols, errors="ignore")
tag_df = tag_df.merge(
    review_df[[REVIEW_ID_COL] + add_cols].drop_duplicates(REVIEW_ID_COL),
    on=REVIEW_ID_COL,
    how="left"
)

In [7]:
# ============================================================
# 플레이 방식 간단 분류
# ============================================================
# Steam categories_text를 이용해 플레이 방식을 크게 2개로 나눈다.
# Co-op, Multi-player, Online 포함: 멀티/협동 요소 포함
# 그 외: Single-player 중심
# 출시 전 체크리스트 조건 필터용 단순 분류다.

review_df["categories_text"] = review_df["categories_text"].fillna("").astype(str)

review_df["play_style"] = "Single-player 중심"

review_df.loc[
    review_df["categories_text"].str.contains("Co-op|Multi-player|Online", case=False, regex=True),
    "play_style"
] = "멀티/협동 요소 포함"

# tag_df도 리뷰 단위로 play_style을 사용할 수 있도록 결합한다.
tag_df = tag_df.merge(
    review_df[[REVIEW_ID_COL, "play_style"]].drop_duplicates(REVIEW_ID_COL),
    on=REVIEW_ID_COL,
    how="left"
)

In [8]:
# ============================================================
# 분석에 필요한 플래그 생성
# ============================================================
# llm_sentiment / tag_sentiment / urgency는 LLM 분석 결과다.
# steam_label_text는 유저가 Steam에서 남긴 추천/비추천 라벨이다.
#
# 우선순위 계산에서는 LLM이 만든 urgency를 직접 기준으로 쓰지 않고,
# Steam 비추천 맥락과 여러 게임 반복성을 우선 기준으로 사용한다.

# 리뷰 단위 플래그
review_df["is_llm_positive"] = review_df["llm_sentiment"].eq("positive")
review_df["is_llm_negative"] = review_df["llm_sentiment"].eq("negative")
review_df["is_llm_mixed"] = review_df["llm_sentiment"].eq("mixed")
review_df["is_high_urgency"] = review_df["llm_urgency_candidate"].eq("high")
review_df["is_steam_positive"] = review_df["steam_label_text"].eq("positive")
review_df["is_steam_negative"] = review_df["steam_label_text"].eq("negative")

# 이슈 태그 단위 플래그
tag_df["is_tag_positive"] = tag_df["llm_issue_sentiment"].eq("positive")
tag_df["is_tag_negative"] = tag_df["llm_issue_sentiment"].eq("negative")
tag_df["is_tag_mixed"] = tag_df["llm_issue_sentiment"].eq("mixed")
tag_df["is_high_urgency"] = tag_df["llm_urgency_candidate"].eq("high")
tag_df["is_steam_positive"] = tag_df["steam_label_text"].eq("positive")
tag_df["is_steam_negative"] = tag_df["steam_label_text"].eq("negative")


# 4. 게임 단위 기본 정보 생성

In [9]:
# ============================================================
# prelaunch_game_base.csv 생성
# ============================================================
# 1행 = 게임 1개
# 나중에 LLM 코드에서 사용자 조건에 해당하는 게임 수를 확인할 때 사용한다.
# 예:
# Action 장르 게임이 몇 개 있는지
# 특정 가격대 게임의 LLM 긍정/부정 비율이 어떤지
# High urgency 후보가 많은 게임이 있는지

game_meta_cols = [
    GAME_ID_COL,
    GAME_NAME_COL,
    "genres_text",
    "price_group",
    "top_steam_tags_text",
    "categories_text",
    "play_style"
]

game_base = (
    review_df
    .groupby(game_meta_cols, dropna=False)
    .agg(
        review_count=(REVIEW_ID_COL, "nunique"),
        llm_positive_count=("is_llm_positive", "sum"),
        llm_negative_count=("is_llm_negative", "sum"),
        llm_mixed_count=("is_llm_mixed", "sum"),
        high_urgency_count=("is_high_urgency", "sum")
    )
    .reset_index()
)

# 게임별 LLM 감정/시급도 비율
# High urgency 비율은 참고 지표이며, 우선순위 계산 기준은 아니다.
game_base["llm_positive_ratio"] = (game_base["llm_positive_count"] / game_base["review_count"] * 100).round(1)
game_base["llm_negative_ratio"] = (game_base["llm_negative_count"] / game_base["review_count"] * 100).round(1)
game_base["high_urgency_ratio"] = (game_base["high_urgency_count"] / game_base["review_count"] * 100).round(1)

# 5. 게임-이슈 단위 내부 집계

이 단계에서 만드는 `game_issue_df`는 내부 계산용 데이터다.  
최종 CSV로 저장하지 않는다.

In [10]:
# ============================================================
# 리뷰-이슈 단위 중복 제거
# ============================================================
# LLM이 한 리뷰 안에서 같은 이슈를 중복 태깅했을 가능성을 줄이기 위해,
# 같은 리뷰 + 같은 이슈 + 같은 이슈 감정 조합은 1번만 본다.
# 이 처리를 하지 않으면 한 리뷰 안의 중복 태그가 전체 반복 이슈처럼 과대 집계될 수 있다.

tag_base = tag_df.dropna(subset=[GAME_ID_COL, REVIEW_ID_COL, ISSUE_COL]).copy()
tag_base = tag_base[tag_base[ISSUE_COL] != ""]
tag_base = tag_base[tag_base[ISSUE_COL].str.lower() != "nan"]

review_issue_df = (
    tag_base
    .drop_duplicates(subset=[REVIEW_ID_COL, ISSUE_COL, "llm_issue_sentiment"])
    .copy()
)

In [11]:
# ============================================================
# 게임-이슈 단위 테이블 생성
# ============================================================
# 같은 게임 안에서 같은 이슈가 여러 리뷰에 반복되어도,
# 최종 중요도 계산에서는 "해당 게임에서 이슈가 발생했다"로 본다.

review_issue_df = review_issue_df.copy()

# 조건부 nunique 계산을 위해 조건에 맞는 review_id만 별도 컬럼으로 만든다.
review_issue_df["steam_positive_review_id"] = np.where(
    review_issue_df["is_steam_positive"], review_issue_df[REVIEW_ID_COL], np.nan
)
review_issue_df["steam_negative_review_id"] = np.where(
    review_issue_df["is_steam_negative"], review_issue_df[REVIEW_ID_COL], np.nan
)
review_issue_df["tag_positive_review_id"] = np.where(
    review_issue_df["is_tag_positive"], review_issue_df[REVIEW_ID_COL], np.nan
)
review_issue_df["tag_negative_review_id"] = np.where(
    review_issue_df["is_tag_negative"], review_issue_df[REVIEW_ID_COL], np.nan
)
review_issue_df["tag_mixed_review_id"] = np.where(
    review_issue_df["is_tag_mixed"], review_issue_df[REVIEW_ID_COL], np.nan
)
review_issue_df["high_urgency_review_id"] = np.where(
    review_issue_df["is_high_urgency"], review_issue_df[REVIEW_ID_COL], np.nan
)

game_issue_df = (
    review_issue_df
    .groupby([GAME_ID_COL, GAME_NAME_COL, ISSUE_COL])
    .agg(
        issue_review_count=(REVIEW_ID_COL, "nunique"),

        # Steam 유저 라벨 기준 맥락
        steam_positive_review_count=("steam_positive_review_id", "nunique"),
        steam_negative_review_count=("steam_negative_review_id", "nunique"),

        # LLM 이슈 감정 기준 보조 지표
        tag_positive_review_count=("tag_positive_review_id", "nunique"),
        tag_negative_review_count=("tag_negative_review_id", "nunique"),
        tag_mixed_review_count=("tag_mixed_review_id", "nunique"),

        # LLM 시급도 후보 보조 지표
        high_urgency_review_count=("high_urgency_review_id", "nunique")
    )
    .reset_index()
)

game_issue_df["issue_present"] = game_issue_df["issue_review_count"] > 0

# 우선순위 계산용 기본 맥락은 Steam 추천/비추천 라벨을 사용한다.
game_issue_df["positive_present"] = game_issue_df["steam_positive_review_count"] > 0
game_issue_df["negative_present"] = game_issue_df["steam_negative_review_count"] > 0

# LLM tag_sentiment는 보조 확인용으로 따로 보관한다.
game_issue_df["tag_positive_present"] = game_issue_df["tag_positive_review_count"] > 0
game_issue_df["tag_negative_present"] = game_issue_df["tag_negative_review_count"] > 0
game_issue_df["tag_mixed_present"] = game_issue_df["tag_mixed_review_count"] > 0

game_issue_df["high_urgency_present"] = game_issue_df["high_urgency_review_count"] > 0

game_issue_df["both_positive_negative_present"] = (
    game_issue_df["positive_present"] & game_issue_df["negative_present"]
)

# 기존 Tableau/보고서 컬럼명과의 호환을 위해 positive/negative_review_count는 Steam 라벨 기준으로 둔다.
game_issue_df["positive_review_count"] = game_issue_df["steam_positive_review_count"]
game_issue_df["negative_review_count"] = game_issue_df["steam_negative_review_count"]
game_issue_df["mixed_review_count"] = game_issue_df["tag_mixed_review_count"]


# 6. 전체 이슈 반복성 요약 생성

이 단계에서는 전체 게임 기준으로 어떤 이슈가 여러 게임에서 반복되는지 확인한다.

중요한 기준은 **리뷰에서 몇 번 언급되었는지**가 아니라, **몇 개 게임에서 반복적으로 나타났는지**다.  
리뷰 수가 많은 일부 게임이 결과를 지배하지 않도록, 게임 단위 반복성을 우선 기준으로 사용한다.

또한 우선순위의 부정 맥락은 가능하면 LLM 판단값보다 더 안정적인 **Steam 추천/비추천 라벨**을 우선 사용한다.  
즉, 어떤 이슈가 여러 게임에서 나타났고 그 게임들에서 Steam 비추천 리뷰와도 연결되는지를 본다.

`High urgency`는 이전 LLM 리뷰 분석에서 생성된 **리뷰 문맥 기반 시급도 후보**다.  
따라서 최종 우선순위를 직접 올리는 기준으로 쓰지 않고, 보조 설명 및 정렬 참고 지표로만 사용한다.


In [12]:
# ============================================================
# 우선순위 함수
# ============================================================
# 이 함수들은 LLM에게 우선순위를 맡기지 않기 위해 사용하는 규칙 기반 함수다.
#
# 04번 출시 후 분석과 구조를 맞춘다.
#
# 핵심 구조:
# 1. get_priority_candidate:
#    - 절대 기준으로 상/중/하 후보를 만든다.
# 2. apply_relative_priority_prelaunch:
#    - 조건별 상대 보정으로 최종 priority_level을 확정한다.
#    - 최종 '상'은 조건별 최대 6개, 전체 개선 대상의 30% 이내로 제한한다.
#
# High urgency는 priority_candidate / priority_level 계산에 직접 사용하지 않는다.
# High urgency는 최종 근거 문장과 대시보드에서 보조 참고 지표로만 남긴다.


def get_priority_candidate(
    issue_game_count,
    issue_game_ratio,
    negative_game_count,
    base_game_count=None
):
    """
    절대 기준 기반 우선순위 후보값을 계산한다.

    주의:
    - 이 함수의 결과는 최종 priority_level이 아니라 priority_candidate로 사용한다.
    - 최종 priority_level은 apply_relative_priority_prelaunch에서 상대 보정을 거친다.
    - High urgency는 LLM 기반 보조 지표이므로 이 함수의 계산식에 넣지 않는다.
    """
    issue_game_count = int(issue_game_count)
    negative_game_count = int(negative_game_count)
    issue_game_ratio = float(issue_game_ratio)

    if base_game_count is None or pd.isna(base_game_count):
        base_game_count = issue_game_count
    else:
        base_game_count = int(base_game_count)

    enough_base_for_high = base_game_count >= HIGH_PRIORITY_MIN_BASE_GAMES
    enough_base_for_mid = base_game_count >= MID_PRIORITY_MIN_BASE_GAMES

    # ------------------------------------------------------------
    # 상 후보
    # ------------------------------------------------------------
    # 여러 게임에서 반복되고, Steam 비추천 맥락도 기준 이상 확인되는 경우
    high_by_count = (
        issue_game_count >= HIGH_PRIORITY_MIN_ISSUE_GAMES
        and negative_game_count >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_GAMES
    )

    # 조건 내 비율이 높더라도,
    # 표본 수, 반복 게임 수, Steam 비추천 맥락이 모두 함께 확인되어야 상 후보로 둔다.
    high_by_ratio = (
        enough_base_for_high
        and issue_game_ratio >= HIGH_PRIORITY_MIN_RATIO
        and issue_game_count >= HIGH_PRIORITY_MIN_ISSUE_GAMES
        and negative_game_count >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_GAMES
    )

    if high_by_count or high_by_ratio:
        return "상"

    # ------------------------------------------------------------
    # 중 후보
    # ------------------------------------------------------------
    # 일부 게임에서 반복되거나, Steam 비추천 맥락이 일정 수준 확인되는 경우
    mid_by_count = issue_game_count >= MID_PRIORITY_MIN_ISSUE_GAMES
    mid_by_negative = negative_game_count >= MID_PRIORITY_MIN_STEAM_NEGATIVE_GAMES
    mid_by_ratio = (
        enough_base_for_mid
        and issue_game_ratio >= MID_PRIORITY_MIN_RATIO
        and issue_game_count >= 2
    )

    if mid_by_count or mid_by_negative or mid_by_ratio:
        return "중"

    return "하"


def get_priority_level(
    issue_game_count,
    issue_game_ratio,
    negative_game_count,
    base_game_count=None
):
    """
    기존 코드 호환용 함수다.
    새 구조에서는 최종 priority_level이 아니라 절대 기준 후보값을 반환한다.
    """
    return get_priority_candidate(
        issue_game_count=issue_game_count,
        issue_game_ratio=issue_game_ratio,
        negative_game_count=negative_game_count,
        base_game_count=base_game_count,
    )


def apply_relative_priority_prelaunch(summary_df, group_cols=None):
    """
    절대 기준 후보값을 바탕으로 최종 상/중/하를 상대적으로 보정한다.

    목적:
    - 조건별로 '상'이 과도하게 많이 나오는 문제를 줄인다.
    - '상'은 해당 조건에서 먼저 확인해야 하는 상위 이슈로 제한한다.
    - 04번 출시 후 분석과 동일하게 '상 후보 → 최종 상 제한 → 나머지는 중 조정' 구조를 사용한다.
    - High urgency는 직접 사용하지 않는다.
    """
    out = summary_df.copy()

    if "priority_candidate" not in out.columns:
        raise ValueError("priority_candidate 컬럼이 필요합니다.")

    out["priority_level"] = "하"
    out["priority_selection_note"] = "절대 기준상 하 또는 점검 우선순위 근거가 상대적으로 약함"

    if group_cols is None:
        group_cols = []

    if isinstance(group_cols, str):
        group_cols = [group_cols]

    # group_cols가 없으면 전체를 하나의 그룹으로 본다.
    if len(group_cols) == 0:
        group_iter = [("all", out.index)]
    else:
        group_iter = out.groupby(group_cols, dropna=False).groups.items()

    for _, group_index in group_iter:
        group_index = list(group_index)
        group = out.loc[group_index].copy()

        # 긍정 칭찬, 기타는 최종 '상' 대상에서 제외한다.
        target_index = group.loc[~group[ISSUE_COL].isin(NO_HIGH_ISSUES)].index
        target_count = len(target_index)

        if target_count == 0:
            continue

        max_high_count = min(
            MAX_HIGH_ISSUES,
            max(1, int(np.ceil(target_count * MAX_HIGH_RATE)))
        )

        # 절대 기준 상 후보 중 최종 '상' 대상만 추린다.
        high_candidates = out.loc[
            target_index[
                out.loc[target_index, "priority_candidate"].eq("상").to_numpy()
            ]
        ].copy()

        if len(high_candidates) > 0:
            # 03-1은 게임 수 기준 분석이므로,
            # Steam 비추천 맥락, 반복 게임 수, 조건 내 비율, 전체 리뷰 수 순서로 상대 우선순위를 정한다.
            high_candidates = high_candidates.sort_values(
                [
                    "negative_game_count",
                    "issue_game_count",
                    "issue_game_ratio",
                    "both_positive_negative_game_count",
                    "total_issue_review_count",
                ],
                ascending=[False, False, False, False, False],
            )

            high_index = high_candidates.head(max_high_count).index

            out.loc[high_index, "priority_level"] = "상"
            out.loc[high_index, "priority_selection_note"] = (
                f"절대 기준 '상' 후보 중 조건 내 상대 우선순위 상위 {max_high_count}개로 유지"
            )

        # 절대 기준에서는 상 후보였지만 최종 상위 이슈에 들지 못한 항목은 중으로 내린다.
        demoted_high_index = target_index[
            (
                out.loc[target_index, "priority_candidate"].eq("상")
                & out.loc[target_index, "priority_level"].ne("상")
            ).to_numpy()
        ]

        out.loc[demoted_high_index, "priority_level"] = "중"
        out.loc[demoted_high_index, "priority_selection_note"] = (
            "절대 기준상 '상' 후보였으나, 최종 '상' 개수 제한 기준에 따라 '중'으로 조정"
        )

        # 절대 기준 중 후보는 중으로 유지한다.
        mid_index = target_index[
            (
                out.loc[target_index, "priority_candidate"].eq("중")
                & out.loc[target_index, "priority_level"].ne("상")
            ).to_numpy()
        ]

        out.loc[mid_index, "priority_level"] = "중"
        out.loc[mid_index, "priority_selection_note"] = "절대 기준상 '중' 후보로 분류"

    # 긍정 칭찬은 강화 요소이므로 최종 점검 우선순위에서는 하로 둔다.
    positive_praise_mask = out[ISSUE_COL].isin(["긍정 칭찬", "positive_praise"])
    out.loc[positive_praise_mask, "priority_level"] = "하"
    out.loc[positive_praise_mask, "priority_selection_note"] = (
        "강점 유지 항목이므로 개선 리스크 우선순위 산정 대상에서 분리"
    )

    # 기타는 원인이 명확하지 않으므로 최종 '상'으로 올리지 않는다.
    other_mask = out[ISSUE_COL].isin(["기타", "other"])
    other_mid_mask = other_mask & out["priority_candidate"].isin(["상", "중"])
    out.loc[other_mid_mask, "priority_level"] = "중"
    out.loc[other_mid_mask, "priority_selection_note"] = (
        "기타 이슈는 원인 범주가 넓어 세부 리뷰 확인 대상으로 분리"
    )

    return out


def get_priority_rule_detail(row):
    """
    priority_level이 어떤 규칙 때문에 부여되었는지 설명용 라벨을 만든다.
    """
    issue_name = str(row.get(ISSUE_COL, ""))
    priority = row["priority_level"]
    candidate = row.get("priority_candidate", priority)

    issue_game_count = int(row["issue_game_count"])
    issue_game_ratio = float(row["issue_game_ratio"])
    negative_game_count = int(row["negative_game_count"])

    base_col = "condition_game_count" if "condition_game_count" in row else "total_game_count"
    base_game_count = int(row[base_col]) if base_col in row and pd.notna(row[base_col]) else issue_game_count

    enough_base_for_mid = base_game_count >= MID_PRIORITY_MIN_BASE_GAMES

    if issue_name in ["긍정 칭찬", "positive_praise"]:
        return "강점 유지 항목이므로 개선 리스크 우선순위 산정 대상에서 분리"

    if issue_name in ["기타", "other"]:
        return "기타 이슈는 원인 범주가 넓어 세부 리뷰 확인 대상으로 분리"

    if priority == "상":
        return "상: 절대 기준 상 후보 중 조건 내 상대 우선순위 상위 이슈"

    if priority == "중" and candidate == "상":
        return "중: 절대 기준 상 후보였으나 최종 상위 이슈 제한 기준에 따라 중으로 조정"

    if priority == "중":
        if (
            issue_game_count >= MID_PRIORITY_MIN_ISSUE_GAMES
            and negative_game_count >= MID_PRIORITY_MIN_STEAM_NEGATIVE_GAMES
        ):
            return "중: 반복 게임 수와 Steam 비추천 맥락이 중간 기준 이상"

        if issue_game_count >= MID_PRIORITY_MIN_ISSUE_GAMES:
            return "중: 반복 게임 수가 중간 기준 이상"

        if negative_game_count >= MID_PRIORITY_MIN_STEAM_NEGATIVE_GAMES:
            return "중: Steam 비추천 맥락이 중간 기준 이상"

        if (
            enough_base_for_mid
            and issue_game_ratio >= MID_PRIORITY_MIN_RATIO
            and issue_game_count >= 2
        ):
            return "중: 조건 내 발생 비율이 중간 기준 이상"

        return "중: 우선순위 규칙 충족"

    return "하: 반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약함"


def get_priority_reason(row):
    """
    LLM 입력과 보고서 설명에 사용할 우선순위 근거 문장을 만든다.
    """
    high_urgency_text = (
        f" High urgency 후보는 {int(row['high_urgency_game_count'])}개 게임에서 확인되었지만, 우선순위 계산에는 직접 사용하지 않고 보조 참고 지표로만 남김."
        if "high_urgency_game_count" in row
        else ""
    )

    candidate_text = (
        f" 절대 기준 후보는 '{row['priority_candidate']}'였음."
        if "priority_candidate" in row
        else ""
    )

    selection_text = (
        f" 최종 보정: {row['priority_selection_note']}."
        if "priority_selection_note" in row
        else ""
    )

    rule_text = (
        f" 적용 규칙: {row['priority_rule_detail']}."
        if "priority_rule_detail" in row
        else ""
    )

    if row["priority_level"] == "상":
        return (
            "여러 게임 반복성, 조건 내 발생 비율, Steam 비추천 맥락을 기준으로 볼 때 "
            "해당 조건에서 우선적으로 확인할 필요가 있는 상위 점검 항목으로 분류함."
            + candidate_text
            + selection_text
            + rule_text
            + high_urgency_text
        )

    if row["priority_level"] == "중":
        return (
            "반복성 또는 Steam 비추천 맥락이 확인되어 출시 전 체크리스트에 포함할 필요가 있는 점검 항목으로 분류함."
            + candidate_text
            + selection_text
            + rule_text
            + high_urgency_text
        )

    return (
        "반복성 또는 Steam 비추천 맥락 근거가 상대적으로 약해 참고용 항목으로 분류함."
        + candidate_text
        + selection_text
        + rule_text
        + high_urgency_text
    )


In [13]:
# ============================================================
# prelaunch_issue_repeat_summary.csv 생성
# ============================================================
# 전체 게임 기준 이슈 반복성 요약표

total_game_count = game_base[GAME_ID_COL].nunique()

issue_repeat_summary = (
    game_issue_df
    .groupby(ISSUE_COL)
    .agg(
        issue_game_count=(GAME_ID_COL, "nunique"),

        # Steam 라벨 기준 맥락
        positive_game_count=("positive_present", "sum"),
        negative_game_count=("negative_present", "sum"),

        # LLM tag_sentiment 기준 보조 맥락
        tag_positive_game_count=("tag_positive_present", "sum"),
        tag_negative_game_count=("tag_negative_present", "sum"),
        tag_mixed_game_count=("tag_mixed_present", "sum"),

        # LLM urgency 기준 보조 맥락
        high_urgency_game_count=("high_urgency_present", "sum"),

        both_positive_negative_game_count=("both_positive_negative_present", "sum"),
        total_issue_review_count=("issue_review_count", "sum")
    )
    .reset_index()
)

issue_repeat_summary["total_game_count"] = total_game_count

issue_repeat_summary["issue_game_ratio"] = (
    issue_repeat_summary["issue_game_count"] / issue_repeat_summary["total_game_count"] * 100
).round(1)

# ------------------------------------------------------------
# 우선순위 계산
# ------------------------------------------------------------
# 1. 절대 기준으로 priority_candidate 생성
# 2. 전체 이슈 기준 상대 보정으로 최종 priority_level 확정

issue_repeat_summary["priority_candidate"] = issue_repeat_summary.apply(
    lambda row: get_priority_candidate(
        row["issue_game_count"],
        row["issue_game_ratio"],
        row["negative_game_count"],
        row["total_game_count"],
    ),
    axis=1
)

issue_repeat_summary = apply_relative_priority_prelaunch(
    issue_repeat_summary,
    group_cols=None
)

issue_repeat_summary["priority_rule_detail"] = issue_repeat_summary.apply(get_priority_rule_detail, axis=1)
issue_repeat_summary["priority_reason"] = issue_repeat_summary.apply(get_priority_reason, axis=1)

# priority_candidate와 priority_selection_note는 계산용 임시 컬럼이다.
# 최종 산출 CSV 구조는 기존과 최대한 동일하게 유지한다.
issue_repeat_summary = issue_repeat_summary.drop(
    columns=["priority_candidate", "priority_selection_note"]
)

issue_repeat_summary["priority_order"] = issue_repeat_summary["priority_level"].map(priority_order_map)

issue_repeat_summary = issue_repeat_summary.sort_values(
    [
        "priority_order",
        "issue_game_count",
        "negative_game_count",
        "issue_game_ratio",
        "total_issue_review_count"
    ],
    ascending=[True, False, False, False, False]
).drop(columns="priority_order")


# 7. 조건별 이슈 요약 생성

이 단계에서는 장르, 가격대, Steam 태그, 플레이 방식별로 반복되는 이슈를 계산한다.

예를 들어 Action 장르에서 어떤 이슈가 여러 게임에 반복적으로 나타나는지,  
10-20 가격대에서 어떤 불만 요인이 자주 나타나는지,  
Roguelike 태그 게임에서 어떤 리스크가 반복되는지를 확인한다.

이 결과는 이후 체크리스트 생성 LLM에 들어가는 핵심 근거가 된다.  
따라서 이 단계에서 상·중·하 우선순위도 LLM이 아니라 **규칙 기반 데이터 기준**으로 먼저 정리한다.

| 우선순위 | 해석 |
|---|---|
| 상 | 여러 게임에서 반복되고, Steam 비추천 맥락도 함께 확인된 이슈 |
| 중 | 일부 게임에서 반복되거나 Steam 비추천 맥락이 일정 수준 확인된 이슈 |
| 하 | 표본이 적거나 반복성·부정 맥락 근거가 약한 참고 이슈 |

`High urgency`는 이전 LLM 분석에서 나온 시급도 후보이므로, `priority_level` 계산에는 직접 사용하지 않는다.  
대신 체크리스트 문장 생성 시 “보조 참고 지표”로만 제공한다.


In [14]:
# ============================================================
# 문자열 리스트 분리 함수
# ============================================================
# "Action, Indie, RPG" 같은 문자열을 ["Action", "Indie", "RPG"] 형태로 바꾼다.
# 예: "Action, Indie, RPG" -> ["Action", "Indie", "RPG"]
# 이 함수는 조건별 게임 목록을 만들기 위한 단순 전처리 함수다.

def split_text_list(text):
    text = str(text)
    text = text.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    values = [value.strip() for value in text.split(",")]
    values = [value for value in values if value and value.lower() != "nan"]
    return values

In [15]:
# ============================================================
# 조건별 게임 목록 생성
# ============================================================
# 1행 = 게임 1개가 어떤 조건값을 가지고 있는지

game_condition_parts = []

# 가격대
price_part = game_base[[GAME_ID_COL, "price_group"]].drop_duplicates().copy()
price_part = price_part.rename(columns={"price_group": "condition_value"})
price_part["condition_type"] = "price_group"
game_condition_parts.append(price_part[[GAME_ID_COL, "condition_type", "condition_value"]])

# 플레이 방식
play_part = game_base[[GAME_ID_COL, "play_style"]].drop_duplicates().copy()
play_part = play_part.rename(columns={"play_style": "condition_value"})
play_part["condition_type"] = "play_style"
game_condition_parts.append(play_part[[GAME_ID_COL, "condition_type", "condition_value"]])

# 장르
genre_part = game_base[[GAME_ID_COL, "genres_text"]].drop_duplicates().copy()
genre_part["condition_value"] = genre_part["genres_text"].apply(split_text_list)
genre_part = genre_part.explode("condition_value")
genre_part["condition_type"] = "genre"
game_condition_parts.append(genre_part[[GAME_ID_COL, "condition_type", "condition_value"]])

# Steam 태그
tag_part = game_base[[GAME_ID_COL, "top_steam_tags_text"]].drop_duplicates().copy()
tag_part["condition_value"] = tag_part["top_steam_tags_text"].apply(split_text_list)
tag_part = tag_part.explode("condition_value")
tag_part["condition_type"] = "steam_tag"
game_condition_parts.append(tag_part[[GAME_ID_COL, "condition_type", "condition_value"]])

game_condition_df = pd.concat(game_condition_parts, ignore_index=True)
game_condition_df["condition_value"] = game_condition_df["condition_value"].astype(str).str.strip()
game_condition_df = game_condition_df[game_condition_df["condition_value"] != ""]
game_condition_df = game_condition_df[game_condition_df["condition_value"].str.lower() != "nan"]

In [16]:
# ============================================================
# 조건별 전체 게임 수 계산
# ============================================================
condition_game_count = (
    game_condition_df
    .groupby(["condition_type", "condition_value"])[GAME_ID_COL]
    .nunique()
    .reset_index(name="condition_game_count")
)

In [17]:
# ============================================================
# prelaunch_condition_issue_summary.csv 생성
# ============================================================
# 1행 = 조건값 + 이슈
# 예: genre=Action에서 UI/UX 이슈가 몇 개 게임에서 반복되었는지

condition_issue_base = game_issue_df.merge(
    game_condition_df,
    on=GAME_ID_COL,
    how="left"
)

condition_issue_summary = (
    condition_issue_base
    .groupby(["condition_type", "condition_value", ISSUE_COL])
    .agg(
        issue_game_count=(GAME_ID_COL, "nunique"),

        # Steam 라벨 기준 맥락
        positive_game_count=("positive_present", "sum"),
        negative_game_count=("negative_present", "sum"),

        # LLM tag_sentiment 기준 보조 맥락
        tag_positive_game_count=("tag_positive_present", "sum"),
        tag_negative_game_count=("tag_negative_present", "sum"),
        tag_mixed_game_count=("tag_mixed_present", "sum"),

        # LLM urgency 기준 보조 맥락
        high_urgency_game_count=("high_urgency_present", "sum"),

        both_positive_negative_game_count=("both_positive_negative_present", "sum"),
        total_issue_review_count=("issue_review_count", "sum")
    )
    .reset_index()
)

condition_issue_summary = condition_issue_summary.merge(
    condition_game_count,
    on=["condition_type", "condition_value"],
    how="left"
)

condition_issue_summary["issue_game_ratio"] = (
    condition_issue_summary["issue_game_count"] / condition_issue_summary["condition_game_count"] * 100
).round(1)

# ------------------------------------------------------------
# 우선순위 계산
# ------------------------------------------------------------
# 1. 절대 기준으로 priority_candidate 생성
# 2. condition_type + condition_value별 상대 보정으로 최종 priority_level 확정
#
# 예:
# - genre=Action 안에서 상위 이슈만 '상'
# - price_group=10-20 안에서 상위 이슈만 '상'
# - steam_tag=Pixel Graphics 안에서 상위 이슈만 '상'
# - play_style=Single-player 중심 안에서 상위 이슈만 '상'

condition_issue_summary["priority_candidate"] = condition_issue_summary.apply(
    lambda row: get_priority_candidate(
        row["issue_game_count"],
        row["issue_game_ratio"],
        row["negative_game_count"],
        row["condition_game_count"],
    ),
    axis=1
)

condition_issue_summary = apply_relative_priority_prelaunch(
    condition_issue_summary,
    group_cols=["condition_type", "condition_value"]
)

condition_issue_summary["priority_rule_detail"] = condition_issue_summary.apply(get_priority_rule_detail, axis=1)
condition_issue_summary["priority_reason"] = condition_issue_summary.apply(get_priority_reason, axis=1)

# priority_candidate와 priority_selection_note는 계산용 임시 컬럼이다.
# 최종 산출 CSV 구조는 기존과 최대한 동일하게 유지한다.
condition_issue_summary = condition_issue_summary.drop(
    columns=["priority_candidate", "priority_selection_note"]
)

condition_issue_summary["priority_order"] = condition_issue_summary["priority_level"].map(priority_order_map)

condition_issue_summary = condition_issue_summary.sort_values(
    [
        "condition_type",
        "condition_value",
        "priority_order",
        "issue_game_count",
        "negative_game_count",
        "issue_game_ratio",
        "total_issue_review_count"
    ],
    ascending=[True, True, True, False, False, False, False]
).drop(columns="priority_order")

# ============================================================
# 이슈 해석 방향 분류
# ============================================================
# priority_level은 점검 중요도이고,
# issue_direction은 이 이슈를 강화 요소로 볼지, 리스크 요소로 볼지 구분한다.

def get_issue_direction(row):
    issue_name = str(row[ISSUE_COL])

    # 명확한 긍정 칭찬 이슈는 강화 요소로 분류
    if issue_name in ["긍정 칭찬"]:
        return "강화 요소"

    # LLM 이슈 감정 기준으로 긍정 맥락이 더 강하면 강화 요소
    if (
        row["tag_positive_game_count"] > row["tag_negative_game_count"]
        and row["tag_positive_game_count"] > row["tag_mixed_game_count"]
    ):
        return "강화 요소"

    # 부정/혼합 맥락이나 Steam 비추천 맥락이 있으면 리스크 요소
    if (
        row["tag_negative_game_count"] > 0
        or row["tag_mixed_game_count"] > 0
        or row["negative_game_count"] > 0
    ):
        return "리스크 요소"

    return "참고 요소"


condition_issue_summary["issue_direction"] = condition_issue_summary.apply(get_issue_direction, axis=1)


# 8. LLM 입력용 근거 문장 데이터 생성

이 단계에서는 조건별 반복 이슈 요약 결과를 LLM이 읽기 쉬운 근거 문장 형태로 바꾼다.

여기서 중요한 점은 LLM에게 우선순위를 새로 판단하게 하지 않는 것이다.  
이전 단계에서 규칙 기반 데이터 기준으로 정리한 우선순위와 근거를 함께 전달하고, LLM은 이를 바탕으로 출시 전 체크리스트 문장을 작성하는 역할만 하게 된다.

근거 문장에는 다음 내용이 포함된다.

| 내용 | 설명 |
|---|---|
| 조건 정보 | 장르, 가격대, Steam 태그, 플레이 방식 중 어떤 조건의 근거인지 |
| 반복 이슈 | 해당 조건에서 여러 게임에 반복적으로 나타난 이슈 |
| Steam 추천/비추천 맥락 | 이 이슈가 Steam 추천/비추천 리뷰가 있는 게임에서 어떻게 나타났는지 |
| LLM 보조 지표 | LLM tag_sentiment, High urgency 분포를 보조 참고로 함께 제공 |
| 데이터 기준 우선순위 | 사전에 정한 규칙으로 계산된 상·중·하 우선순위 |
| 우선순위 적용 규칙 | 해당 이슈가 왜 상·중·하로 분류되었는지 설명하는 규칙 라벨 |



In [18]:
# ============================================================
# LLM 근거 문장 생성 함수
# ============================================================
# 나중에 LLM 프롬프트에 바로 넣기 쉬운 문장을 만든다.
# High urgency는 우선순위 직접 기준이 아니라 보조 참고 지표라고 명시한다.

def make_evidence_text(row):
    text = (
        f"{row['condition_type']} 조건 '{row['condition_value']}'에서 "
        f"'{row[ISSUE_COL]}' 항목은 전체 {int(row['condition_game_count'])}개 게임 중 "
        f"{int(row['issue_game_count'])}개 게임에서 반복되었다"
        f"({row['issue_game_ratio']}%). "
        f"Steam 비추천 맥락은 {int(row['negative_game_count'])}개 게임, "
        f"Steam 추천 맥락은 {int(row['positive_game_count'])}개 게임에서 확인되었다. "
        f"규칙 기반 점검 중요도는 '{row['priority_level']}'이며, "
        f"적용된 기준은 '{row['priority_rule_detail']}'이다. "
        f"LLM이 High urgency로 분류한 사례는 {int(row['high_urgency_game_count'])}개 게임에서 나타났지만, "
        f"이는 점검 중요도 계산에 직접 사용하지 않은 보조 참고 지표다."
    )
    return text


In [19]:
# ============================================================
# prelaunch_checklist_evidence_base.csv 생성
# ============================================================
# 조건별 이슈 요약 중에서 LLM에게 넘기기 좋은 형태만 정리한다.
# 사용자 조건 필터링은 나중에 LLM 코드에서 수행한다.

checklist_evidence_base = condition_issue_summary.copy()

checklist_evidence_base["llm_evidence_text"] = checklist_evidence_base.apply(make_evidence_text, axis=1)

# 조건별로 너무 많은 이슈가 들어가지 않도록 상위 이슈만 남긴다.
# 따라서 prelaunch_checklist_evidence_base.csv는 전체 이슈 목록이 아니라
# 03-2 체크리스트 생성을 위한 대표 근거 데이터다.
# 전체 조건별 이슈 목록은 prelaunch_condition_issue_summary.csv에 보존한다.
checklist_evidence_base["priority_order"] = checklist_evidence_base["priority_level"].map(priority_order_map)

checklist_evidence_base = (
    checklist_evidence_base
    .sort_values(
        [
            "condition_type",
            "condition_value",
            "priority_order",
            "issue_game_count",
            "negative_game_count",
            "issue_game_ratio",
            "total_issue_review_count"
        ],
        ascending=[True, True, True, False, False, False, False]
    )
    .groupby(["condition_type", "condition_value"])
    .head(TOP_N_EVIDENCE_PER_CONDITION)
    .reset_index(drop=True)
    .drop(columns="priority_order")
)

checklist_evidence_base = checklist_evidence_base[
    [
        "condition_type",
        "condition_value",
        ISSUE_COL,
        "issue_direction",
        "condition_game_count",
        "issue_game_count",
        "issue_game_ratio",
        "positive_game_count",
        "negative_game_count",
        "tag_positive_game_count",
        "tag_negative_game_count",
        "tag_mixed_game_count",
        "high_urgency_game_count",
        "both_positive_negative_game_count",
        "total_issue_review_count",
        "priority_level",
        "priority_rule_detail",
        "priority_reason",
        "llm_evidence_text"
    ]
]


# 9. 우선순위 산정 로직 점검

이 단계에서는 `priority_level`이 LLM의 `High urgency` 값으로 직접 결정되지 않는지 확인한다.

점검 내용은 다음과 같다.

| 점검 항목 | 의미 |
|---|---|
| `get_priority_level` 입력값 확인 | 함수 인자에 `high_urgency_game_count`가 없는지 확인 |
| `상` 우선순위 조건 확인 | `상`으로 분류된 이슈가 Steam 비추천 맥락 없이 올라가지 않았는지 확인 |
| 우선순위 분포 확인 | 상·중·하 분포를 간단히 확인 |


In [20]:
# ============================================================
# 우선순위 산정 로직 검증
# ============================================================
# 목적:
# 04번 출시 후 분석과 동일한 검증 철학을 사용하되,
# 03-1 출시 전 분석의 "게임 수 기반 반복성" 구조에 맞게 점검한다.
#
# 주의:
# - 03-1은 출시 전 체크리스트 근거 생성 코드다.
# - 따라서 출시 후 분석의 "즉시 확인 / 저장·진행 blocking 예외" 기준은 적용하지 않는다.
# - priority_level은 LLM High urgency가 아니라,
#   반복 게임 수 + 조건 내 발생 비율 + Steam 비추천 맥락 + 상대 보정으로 계산되어야 한다.

# ------------------------------------------------------------
# 0. 검증용 함수
# ------------------------------------------------------------
FORBIDDEN_PRIORITY_REFS = {
    "high_urgency",
    "high_urgency_game_count",
    "high_urgency_review_count",
}


def _find_forbidden_refs(func, forbidden_refs):
    func_names = {str(x) for x in func.__code__.co_names}
    func_consts = {
        str(x)
        for x in func.__code__.co_consts
        if isinstance(x, str)
    }
    func_refs = func_names | func_consts

    return [
        ref
        for ref in func_refs
        if any(forbidden in ref for forbidden in forbidden_refs)
    ]


def _candidate_high_rule_basis(df, base_col):
    """
    get_priority_candidate 함수의 '상 후보' 기준을 검증용으로 재현한다.
    High urgency는 포함하지 않는다.
    """
    high_by_count = (
        (df["issue_game_count"] >= HIGH_PRIORITY_MIN_ISSUE_GAMES)
        & (df["negative_game_count"] >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_GAMES)
    )

    high_by_ratio = (
        (df[base_col] >= HIGH_PRIORITY_MIN_BASE_GAMES)
        & (df["issue_game_ratio"] >= HIGH_PRIORITY_MIN_RATIO)
        & (df["issue_game_count"] >= HIGH_PRIORITY_MIN_ISSUE_GAMES)
        & (df["negative_game_count"] >= HIGH_PRIORITY_MIN_STEAM_NEGATIVE_GAMES)
    )

    return high_by_count | high_by_ratio


def _audit_priority_table(df, table_name, base_col, group_cols=None):
    """
    priority_level 검증 공통 함수.
    """
    if group_cols is None:
        group_cols = []

    if isinstance(group_cols, str):
        group_cols = [group_cols]

    print(f"\n[{table_name}] 검증 시작")

    # 1. 필수 컬럼 확인
    required_cols = {
        ISSUE_COL,
        "issue_game_count",
        "negative_game_count",
        "issue_game_ratio",
        "priority_level",
        base_col,
    }

    missing_cols = sorted(required_cols - set(df.columns))
    assert len(missing_cols) == 0, f"{table_name} 필수 컬럼 누락: {missing_cols}"

    # 2. issue_game_count는 base_col보다 클 수 없다.
    over_base = df[df["issue_game_count"] > df[base_col]]
    assert len(over_base) == 0, over_base[
        [ISSUE_COL, "issue_game_count", base_col, "priority_level"]
    ]

    # 3. negative_game_count는 issue_game_count보다 클 수 없다.
    over_issue_count = df[df["negative_game_count"] > df["issue_game_count"]]
    assert len(over_issue_count) == 0, over_issue_count[
        [ISSUE_COL, "issue_game_count", "negative_game_count", "priority_level"]
    ]

    # 4. 최종 '상'은 반드시 get_priority_candidate의 '상 후보' 기준을 만족해야 한다.
    candidate_high_has_rule_basis = _candidate_high_rule_basis(df, base_col)

    high_without_rule_basis = df[
        (df["priority_level"] == "상")
        & ~candidate_high_has_rule_basis
    ]

    display_cols = [
        col for col in [
            "condition_type",
            "condition_value",
            ISSUE_COL,
            base_col,
            "issue_game_count",
            "issue_game_ratio",
            "negative_game_count",
            "high_urgency_game_count",
            "priority_level",
            "priority_rule_detail",
        ]
        if col in df.columns
    ]

    assert len(high_without_rule_basis) == 0, high_without_rule_basis[display_cols]

    # 5. Steam 비추천 맥락 없이 '상'으로 올라가면 안 된다.
    high_without_negative = df[
        (df["priority_level"] == "상")
        & (df["negative_game_count"] < HIGH_PRIORITY_MIN_STEAM_NEGATIVE_GAMES)
    ]

    assert len(high_without_negative) == 0, high_without_negative[display_cols]

    # 6. 긍정 칭찬/기타는 최종 '상'으로 올라가면 안 된다.
    no_high_issue_check = df[
        (df[ISSUE_COL].isin(NO_HIGH_ISSUES))
        & (df["priority_level"] == "상")
    ]

    assert len(no_high_issue_check) == 0, no_high_issue_check[display_cols]

    # 7. 최종 '상' 개수 제한 확인
    if len(group_cols) == 0:
        priority_target_count = int((~df[ISSUE_COL].isin(NO_HIGH_ISSUES)).sum())
        max_allowed_high_count = (
            min(MAX_HIGH_ISSUES, max(1, int(np.ceil(priority_target_count * MAX_HIGH_RATE))))
            if priority_target_count > 0
            else 0
        )
        actual_high_count = int((df["priority_level"] == "상").sum())

        assert actual_high_count <= max_allowed_high_count, (
            f"{table_name} 최종 '상' 이슈 수가 제한 기준을 초과했습니다: "
            f"{actual_high_count} > {max_allowed_high_count}"
        )

        high_limit_summary = pd.DataFrame([{
            "table_name": table_name,
            "priority_target_count": priority_target_count,
            "high_count": actual_high_count,
            "max_allowed_high_count": max_allowed_high_count,
        }])

    else:
        high_limit_summary = (
            df
            .assign(is_priority_target=~df[ISSUE_COL].isin(NO_HIGH_ISSUES))
            .groupby(group_cols, dropna=False)
            .agg(
                priority_target_count=("is_priority_target", "sum"),
                high_count=("priority_level", lambda x: (x == "상").sum())
            )
            .reset_index()
        )

        high_limit_summary["max_allowed_high_count"] = high_limit_summary["priority_target_count"].apply(
            lambda x: min(MAX_HIGH_ISSUES, max(1, int(np.ceil(x * MAX_HIGH_RATE)))) if x > 0 else 0
        )

        over_high_limit = high_limit_summary[
            high_limit_summary["high_count"] > high_limit_summary["max_allowed_high_count"]
        ]

        assert len(over_high_limit) == 0, over_high_limit

    # 8. 우선순위별 주요 지표 분포 출력
    audit_summary = (
        df
        .groupby("priority_level")
        .agg(
            row_count=(ISSUE_COL, "count"),
            avg_issue_game_count=("issue_game_count", "mean"),
            avg_negative_game_count=("negative_game_count", "mean"),
            avg_high_urgency_game_count=("high_urgency_game_count", "mean"),
        )
        .reindex(["상", "중", "하"])
        .round(2)
        .reset_index()
    )

    print(f"[{table_name}] 검증 통과")
    print("우선순위 분포:")
    display(
        df["priority_level"]
        .value_counts()
        .rename_axis("priority_level")
        .reset_index(name="row_count")
    )

    print("최종 '상' 제한 기준 확인:")
    display(high_limit_summary)

    print("우선순위별 주요 지표 분포:")
    display(audit_summary)


# ------------------------------------------------------------
# 1. priority 계산 함수에서 High urgency를 직접 참조하지 않는지 확인
# ------------------------------------------------------------
candidate_forbidden_refs = _find_forbidden_refs(
    get_priority_candidate,
    FORBIDDEN_PRIORITY_REFS
)
assert len(candidate_forbidden_refs) == 0, (
    "get_priority_candidate 계산 함수에서 High urgency 계열 참조가 발견되었습니다: "
    f"{candidate_forbidden_refs}"
)

relative_forbidden_refs = _find_forbidden_refs(
    apply_relative_priority_prelaunch,
    FORBIDDEN_PRIORITY_REFS
)
assert len(relative_forbidden_refs) == 0, (
    "apply_relative_priority_prelaunch 계산 함수에서 High urgency 계열 참조가 발견되었습니다: "
    f"{relative_forbidden_refs}"
)

# ------------------------------------------------------------
# 2. 전체 이슈 반복 요약 검증
# ------------------------------------------------------------
_audit_priority_table(
    df=issue_repeat_summary,
    table_name="prelaunch_issue_repeat_summary",
    base_col="total_game_count",
    group_cols=None,
)

# ------------------------------------------------------------
# 3. 조건별 이슈 반복 요약 검증
# ------------------------------------------------------------
_audit_priority_table(
    df=condition_issue_summary,
    table_name="prelaunch_condition_issue_summary",
    base_col="condition_game_count",
    group_cols=["condition_type", "condition_value"],
)

# ------------------------------------------------------------
# 4. 03-2 체크리스트 입력용 근거 텍스트 검증
# ------------------------------------------------------------
evidence_empty = checklist_evidence_base[
    checklist_evidence_base["llm_evidence_text"].isna()
    | checklist_evidence_base["llm_evidence_text"].astype(str).str.strip().eq("")
]

assert len(evidence_empty) == 0, evidence_empty[
    [
        "condition_type",
        "condition_value",
        ISSUE_COL,
        "priority_level",
        "llm_evidence_text",
    ]
]

print("\n검증 통과")
print("High urgency는 priority_level 계산에는 직접 사용하지 않고, priority_reason/llm_evidence_text에서 보조 지표로만 유지한다.")



[prelaunch_issue_repeat_summary] 검증 시작
[prelaunch_issue_repeat_summary] 검증 통과
우선순위 분포:


,priority_level,row_count
0,중,14
1,상,6
2,하,1


최종 '상' 제한 기준 확인:


,table_name,priority_target_count,high_count,max_allowed_high_count
0,prelaunch_issue_repeat_summary,19,6,6


우선순위별 주요 지표 분포:


...

최종 '상' 제한 기준 확인:


,condition_type,condition_value,priority_target_count,high_count,max_allowed_high_count
0,genre,Action,19,6,6
1,genre,Adventure,19,6,6
2,genre,Casual,19,6,6
3,genre,Indie,19,6,6
4,genre,RPG,19,6,6
...,...,...,...,...,...
261,steam_tag,Wholesome,1,0,1
262,steam_tag,Word Game,9,0,3
263,steam_tag,World War I,2,0,1
264,steam_tag,World War II,9,0,3


우선순위별 주요 지표 분포:


,priority_level,row_count,avg_issue_game_count,avg_negative_game_count,avg_high_urgency_game_count
0,상,312,15.90,9.46,7.79
1,중,1470,5.23,2.68,2.51
2,하,2042,1.82,0.72,0.72



검증 통과
High urgency는 priority_level 계산에는 직접 사용하지 않고, priority_reason/llm_evidence_text에서 보조 지표로만 유지한다.


# 10. CSV 저장


In [21]:
# ============================================================
# CSV 저장
# ============================================================
# 최종 저장 파일은 4개만 만든다.

game_base.to_csv(GAME_BASE_PATH, index=False, encoding="utf-8-sig")
issue_repeat_summary.to_csv(ISSUE_REPEAT_SUMMARY_PATH, index=False, encoding="utf-8-sig")
condition_issue_summary.to_csv(CONDITION_ISSUE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
checklist_evidence_base.to_csv(CHECKLIST_EVIDENCE_BASE_PATH, index=False, encoding="utf-8-sig")

print("CSV 저장 완료")
print(GAME_BASE_PATH)
print(ISSUE_REPEAT_SUMMARY_PATH)
print(CONDITION_ISSUE_SUMMARY_PATH)
print(CHECKLIST_EVIDENCE_BASE_PATH)

CSV 저장 완료
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\prelaunch_checklist_data\prelaunch_game_base.csv
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\prelaunch_checklist_data\prelaunch_issue_repeat_summary.csv
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\prelaunch_checklist_data\prelaunch_condition_issue_summary.csv
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\prelaunch_checklist_data\prelaunch_checklist_evidence_base.csv


# 11. Tableau 대시보드용 소스 CSV 생성

이번 단계에서는 조원이 Tableau에서 바로 사용할 수 있도록 원천/요약 소스 CSV를 따로 저장한다.

Tableau에서 가장 먼저 쓸 메인 파일은 `tableau_allgames_prelaunch_source.csv`다.


In [22]:
# ============================================================
# 10. Tableau 대시보드용 단일 CSV 생성
# ============================================================

TABLEAU_DIR = MASTER_DIR / "tableau_dashboard_csv"
TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

TABLEAU_SOURCE_PATH = TABLEAU_DIR / "tableau_allgames_prelaunch_source.csv"

print("Tableau 저장 폴더:", TABLEAU_DIR)
print("Tableau 저장 파일:", TABLEAU_SOURCE_PATH)

Tableau 저장 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\tableau_dashboard_csv
Tableau 저장 파일: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\tableau_dashboard_csv\tableau_allgames_prelaunch_source.csv


In [23]:
# ============================================================
# 10-2. 게임-조건-이슈 단위 Tableau 원천 데이터 생성
# ============================================================
# 데이터 단위:
# 1행 = 특정 게임(appid)이 특정 조건값을 가지고 있고,
#       그 게임에서 특정 이슈가 발생한 경우
#
# 예:
# - appid=123, condition_type=genre, condition_value=Action, issue_name_kor=UI/UX
# - appid=123, condition_type=steam_tag, condition_value=Roguelike, issue_name_kor=난이도

tableau_source = game_issue_df.merge(
    game_condition_df,
    on=GAME_ID_COL,
    how="left"
)

# 게임 기본 정보 결합
game_info_cols = [
    GAME_ID_COL,
    GAME_NAME_COL,
    "genres_text",
    "price_group",
    "top_steam_tags_text",
    "categories_text",
    "play_style",
    "review_count",
    "llm_positive_count",
    "llm_negative_count",
    "llm_mixed_count",
    "high_urgency_count",
    "llm_positive_ratio",
    "llm_negative_ratio",
    "high_urgency_ratio",
]

tableau_source = tableau_source.merge(
    game_base[game_info_cols],
    on=[GAME_ID_COL, GAME_NAME_COL],
    how="left"
)


In [24]:
# ============================================================
# 10-3. 조건별 집계값 결합
# ============================================================
# Tableau에서 바로 사용할 수 있도록 조건별 반복 이슈 요약값을 붙인다.
# 단, priority_reason 같은 문장형 설명 컬럼은 제외한다.

condition_summary_for_tableau = condition_issue_summary[
    [
        "condition_type",
        "condition_value",
        ISSUE_COL,
        "condition_game_count",
        "issue_game_count",
        "issue_game_ratio",
        "positive_game_count",
        "negative_game_count",
        "tag_positive_game_count",
        "tag_negative_game_count",
        "tag_mixed_game_count",
        "high_urgency_game_count",
        "both_positive_negative_game_count",
        "priority_level",
        "priority_rule_detail",
    ]
].copy()

condition_summary_for_tableau["priority_order"] = (
    condition_summary_for_tableau["priority_level"].map(priority_order_map)
)

tableau_source = tableau_source.merge(
    condition_summary_for_tableau,
    on=["condition_type", "condition_value", ISSUE_COL],
    how="left"
)

# Tableau에서 보기 쉬운 한글 조건명
condition_type_kor_map = {
    "genre": "장르",
    "price_group": "가격대",
    "steam_tag": "Steam 태그",
    "play_style": "플레이 방식",
}

tableau_source["condition_type_kor"] = tableau_source["condition_type"].map(condition_type_kor_map)


In [25]:
# ============================================================
# 11-4. Steam 태그 DNA 참고 정보 결합
# ============================================================
# 조원 EDA 방식:
# 성과 상위권 게임에서 어떤 Steam 태그가 자주 나타나는지 확인한다.
#
# 이 정보는 "이 태그가 성공을 보장한다"는 의미가 아니다.
# Steam 태그 조건을 해석할 때 참고할 수 있는 보조 맥락으로만 사용한다.
#
# 이 컬럼들은 condition_type == "steam_tag" 행에서만 값이 들어간다.

def parse_tag_keys(tag_text):
    # steam_indie_games_graded.csv의 tags 문자열에서 태그명만 추출한다.
    if pd.isna(tag_text):
        return []

    text = str(tag_text).strip()
    if text == "" or text.lower() == "nan":
        return []

    parsed = ast.literal_eval(text)

    # 현재 데이터의 tags는 {태그명: 투표수} 형태의 문자열 dict다.
    if isinstance(parsed, dict):
        return [str(tag).strip() for tag in parsed.keys()]

    # 혹시 리스트 형태로 저장된 경우에도 태그명 리스트로 맞춘다.
    if isinstance(parsed, list):
        return [str(tag).strip() for tag in parsed]

    return []


graded_games = pd.read_csv(GRADED_GAMES_PATH)

graded_games["steam_tag"] = graded_games["tags"].apply(parse_tag_keys)

tag_dna_base = (
    graded_games[
        [
            GAME_ID_COL,
            "name",
            "performance_grade",
            "scale_grade",
            "satisfaction_grade",
            "positive_rate",
            "total_reviews",
            "steam_tag",
        ]
    ]
    .explode("steam_tag")
    .rename(columns={"name": "graded_game_name"})
)

tag_dna_base["steam_tag"] = tag_dna_base["steam_tag"].astype(str).str.strip()
tag_dna_base = tag_dna_base[tag_dna_base["steam_tag"] != ""]
tag_dna_base = tag_dna_base[tag_dna_base["steam_tag"].str.lower() != "nan"]

# 성과 상위권으로 볼 등급값
# 프로젝트에서 사용한 performance_grade 체계가 혼재될 수 있어 한글 설명 대신 코드값만 정리한다.
top_performance_grades = [
    "high_high",
    "high_mid",
    "mid_high",
    "HH",
    "HM",
    "MH",
]

tag_dna_base["is_top_performance_group"] = (
    tag_dna_base["performance_grade"].isin(top_performance_grades).astype(int)
)

# 태그별로 전체 등장 게임 수, 상위 성과권 게임 수, 평균 긍정률, 중앙 리뷰 수를 요약한다.
tag_dna_summary = (
    tag_dna_base
    .groupby("steam_tag")
    .agg(
        tag_dna_game_count=(GAME_ID_COL, "nunique"),
        top_performance_game_count=("is_top_performance_group", "sum"),
        avg_positive_rate=("positive_rate", "mean"),
        median_total_reviews=("total_reviews", "median"),
    )
    .reset_index()
)

tag_dna_summary["top_performance_ratio"] = (
    tag_dna_summary["top_performance_game_count"]
    / tag_dna_summary["tag_dna_game_count"]
    * 100
).round(1)

tag_dna_summary["avg_positive_rate"] = tag_dna_summary["avg_positive_rate"].round(1)
tag_dna_summary["median_total_reviews"] = tag_dna_summary["median_total_reviews"].round(0)

tableau_source = tableau_source.merge(
    tag_dna_summary,
    left_on="condition_value",
    right_on="steam_tag",
    how="left"
)

# Steam 태그 조건이 아닌 행에서는 태그 DNA 값을 비워둔다.
tag_dna_cols = [
    "tag_dna_game_count",
    "top_performance_game_count",
    "top_performance_ratio",
    "avg_positive_rate",
    "median_total_reviews",
]

for col in tag_dna_cols:
    tableau_source.loc[tableau_source["condition_type"] != "steam_tag", col] = np.nan

tableau_source = tableau_source.drop(columns=["steam_tag"], errors="ignore")

In [26]:
# ============================================================
# 11-5. Tableau용 최종 컬럼 정리 및 저장
# ============================================================
# priority_reason, tableau_note 같은 문장형 설명 컬럼은 제외한다.
# Tableau에서는 숫자/범주형 컬럼 중심으로 필터와 그래프를 구성한다.

tableau_cols = [
    # 게임 식별 정보
    GAME_ID_COL,
    GAME_NAME_COL,

    # 조건 정보
    "condition_type",
    "condition_type_kor",
    "condition_value",

    # 게임 메타 정보
    "genres_text",
    "price_group",
    "top_steam_tags_text",
    "categories_text",
    "play_style",

    # 이슈 정보
    ISSUE_COL,

    # 게임 내부 이슈 발생 정보
    "issue_review_count",
    "positive_review_count",
    "negative_review_count",
    "mixed_review_count",
    "high_urgency_review_count",
    "tag_positive_present",
    "tag_negative_present",
    "tag_mixed_present",
    "high_urgency_present",
    "both_positive_negative_present",

    # 게임 단위 리뷰 요약
    "review_count",
    "llm_positive_count",
    "llm_negative_count",
    "llm_mixed_count",
    "high_urgency_count",
    "llm_positive_ratio",
    "llm_negative_ratio",
    "high_urgency_ratio",

    # 조건별 반복 이슈 집계값
    "condition_game_count",
    "issue_game_count",
    "issue_game_ratio",
    "positive_game_count",
    "negative_game_count",
    "tag_positive_game_count",
    "tag_negative_game_count",
    "tag_mixed_game_count",
    "high_urgency_game_count",
    "both_positive_negative_game_count",
    "priority_level",
    "priority_rule_detail",
    "priority_order",

    # Steam 태그 DNA 참고 정보
    "tag_dna_game_count",
    "top_performance_game_count",
    "top_performance_ratio",
    "avg_positive_rate",
    "median_total_reviews",
]

tableau_source = tableau_source[tableau_cols].copy()

tableau_source.to_csv(
    TABLEAU_SOURCE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Tableau 대시보드용 단일 CSV 저장 완료")
print(TABLEAU_SOURCE_PATH)
print("데이터 크기:", tableau_source.shape)


Tableau 대시보드용 단일 CSV 저장 완료
c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\prelaunch\master\tableau_dashboard_csv\tableau_allgames_prelaunch_source.csv
데이터 크기: (16741, 47)


In [27]:
tableau_source.head()

,appid,game_name,condition_type,condition_type_kor,condition_value,genres_text,price_group,top_steam_tags_text,categories_text,play_style,issue_name_kor,issue_review_count,positive_review_count,negative_review_count,mixed_review_count,high_urgency_review_count,tag_positive_present,tag_negative_present,tag_mixed_present,high_urgency_present,both_positive_negative_present,review_count,llm_positive_count,llm_negative_count,llm_mixed_count,high_urgency_count,llm_positive_ratio,llm_negative_ratio,high_urgency_ratio,condition_game_count,issue_game_count,issue_game_ratio,positive_game_count,negative_game_count,tag_positive_game_count,tag_negative_game_count,tag_mixed_game_count,high_urgency_game_count,both_positive_negative_game_count,priority_level,priority_rule_detail,priority_order,tag_dna_game_count,top_performance_game_count,top_performance_ratio,avg_positive_rate,median_total_reviews
0,571740,Golf It!,price_group,가격대,0-5,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,2,2,0,2,False,True,False,True,True,48,27,15,4,9,56.2,31.2,18.8,59.0,25.0,42.4,21.0,12.0,7.0,19.0,0.0,12.0,8.0,상,상: 절대 기준 상 후보 중 조건 내 상대 우선순위 상위 이슈,1.0,NaN,NaN,NaN,NaN,NaN
1,571740,Golf It!,play_style,플레이 방식,멀티/협동 요소 포함,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,2,2,0,2,False,True,False,True,True,48,27,15,4,9,56.2,31.2,18.8,31.0,19.0,61.3,19.0,13.0,9.0,17.0,1.0,12.0,13.0,상,상: 절대 기준 상 후보 중 조건 내 상대 우선순위 상위 이슈,1.0,NaN,NaN,NaN,NaN,NaN
2,571740,Golf It!,genre,장르,Casual,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,2,2,0,2,False,True,False,True,True,48,27,15,4,9,56.2,31.2,18.8,73.0,37.0,50.7,34.0,19.0,13.0,30.0,1.0,17.0,16.0,상,상: 절대 기준 상 후보 중 조건 내 상대 우선순위 상위 이슈,1.0,NaN,NaN,NaN,NaN,NaN
3,571740,Golf It!,genre,장르,Indie,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,2,2,0,2,False,True,False,True,True,48,27,15,4,9,56.2,31.2,18.8,163.0,95.0,58.3,84.0,58.0,26.0,83.0,4.0,51.0,47.0,상,상: 절대 기준 상 후보 중 조건 내 상대 우선순위 상위 이슈,1.0,NaN,NaN,NaN,NaN,NaN
4,571740,Golf It!,genre,장르,Simulation,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplayer, Simulation, Level Editor","Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-Platform Multiplayer, Steam Achievement...",멀티/협동 요소 포함,UI/UX,4,2,2,0,2,False,True,False,True,True,48,27,15,4,9,56.2,31.2,18.8,40.0,29.0,72.5,26.0,23.0,9.0,27.0,1.0,17.0,20.0,상,상: 절대 기준 상 후보 중 조건 내 상대 우선순위 상위 이슈,1.0,NaN,NaN,NaN,NaN,NaN


# 12. 출시 전 LLM 분석 산출 CSV 테이블 명세서


## 산출 파일 요약
| 파일명 | 데이터 단위 | 주요 역할 |
|---|---|---|
| `prelaunch_game_base.csv` | 1행 = 게임 1개 | LLM 분석에 사용된 게임별 기본 메타데이터와 LLM 감정/시급도 요약을 저장한다. |
| `prelaunch_issue_repeat_summary.csv` | 1행 = 이슈 1개 | 전체 게임 기준으로 어떤 이슈가 여러 게임에서 반복되는지 요약한다. |
| `prelaunch_condition_issue_summary.csv` | 1행 = 조건값 1개 + 이슈 1개 | 장르·가격대·Steam 태그·플레이 방식별로 반복 이슈와 규칙 기반 우선순위를 정리한다. |
| `prelaunch_checklist_evidence_base.csv` | 1행 = 조건값 1개 + 이슈 1개 | 조건별 이슈 요약 결과를 LLM이 읽기 쉬운 근거 문장 형태로 변환한 테이블이다. |
| `tableau_allgames_prelaunch_source.csv` | 1행 = 게임 1개 + 조건값 1개 + 이슈 1개 | 게임 정보, 조건별 이슈 요약, 우선순위, Steam 태그 DNA 참고 지표를 한 파일에 통합한 Tableau용 데이터다. |

## prelaunch_game_base.csv
- 설명: 게임 1개 단위 요약 테이블

| 컬럼명 | 설명 | 예시 값 |
| --- | --- | --- |
| `appid` | Steam 게임 고유 ID | 571740 |
| `game_name` | Steam 게임명 | Golf It! |
| `genres_text` | 게임의 Steam 장르 목록을 쉼표로 연결한 값 | Casual, Indie, Simulation, Sports |
| `price_group` | 게임 가격을 분석용 구간으로 나눈 값 | 0-5 |
| `top_steam_tags_text` | 게임의 주요 Steam 태그 목록을 쉼표로 연결한 값 | Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplay... |
| `categories_text` | Steam 카테고리 목록을 쉼표로 연결한 값 | Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-... |
| `play_style` | 카테고리 정보를 바탕으로 분류한 플레이 방식 | 멀티/협동 요소 포함 |
| `review_count` | 해당 게임에서 LLM 분석에 사용된 리뷰 수 | 48 |
| `llm_positive_count` | LLM이 긍정으로 분류한 리뷰 수 | 27 |
| `llm_negative_count` | LLM이 부정으로 분류한 리뷰 수 | 15 |
| `llm_mixed_count` | LLM이 혼합으로 분류한 리뷰 수 | 4 |
| `high_urgency_count` | LLM이 High urgency로 분류한 리뷰 수 | 9 |
| `llm_positive_ratio` | LLM 긍정 리뷰 비율 | 56.2 |
| `llm_negative_ratio` | LLM 부정 리뷰 비율 | 31.2 |
| `high_urgency_ratio` | LLM High urgency 리뷰 비율 | 18.8 |


## prelaunch_issue_repeat_summary.csv

- 설명: 전체 게임 기준 이슈 반복 요약 테이블

| 컬럼명 | 설명 | 예시 값 |
| --- | --- | --- |
| `issue_name_kor` | LLM 이슈 태그를 한국어 분석명으로 정리한 값 | 긍정 칭찬 |
| `issue_game_count` | 해당 이슈가 1회 이상 등장한 게임 수 | 140 |
| `positive_game_count` | 해당 이슈가 Steam 추천 리뷰 맥락에서 등장한 게임 수 | 139 |
| `negative_game_count` | 해당 이슈가 Steam 비추천 리뷰 맥락에서 등장한 게임 수 | 28 |
| `tag_positive_game_count` | LLM이 해당 이슈를 긍정 태그로 분류한 게임 수 | 140 |
| `tag_negative_game_count` | LLM이 해당 이슈를 부정 태그로 분류한 게임 수 | 0 |
| `tag_mixed_game_count` | LLM이 해당 이슈를 혼합 태그로 분류한 게임 수 | 0 |
| `high_urgency_game_count` | 해당 이슈가 High urgency 리뷰에서 등장한 게임 수 | 31 |
| `both_positive_negative_game_count` | 해당 이슈가 추천/비추천 맥락 모두에서 등장한 게임 수 | 27 |
| `total_issue_review_count` | 해당 이슈가 등장한 전체 리뷰 수 | 2719 |
| `total_game_count` | 분석 대상 전체 게임 수 | 151 |
| `issue_game_ratio` | 조건 내 게임 중 해당 이슈가 등장한 게임 비율 | 92.7 |
| `priority_level` | 반복성 및 Steam 비추천 맥락 기준으로 계산한 규칙 기반 점검 중요도. 개선 우선순위가 아니라 출시 전 확인 우선도에 가까움 | 상 |
| `priority_rule_detail` | 우선순위 산정에 적용된 규칙 설명 | 상: 반복 게임 수와 Steam 비추천 맥락이 모두 기준 이상 |
| `priority_reason` | 우선순위가 부여된 이유를 문장으로 정리한 값 | 여러 게임 반복성, 조건 내 발생 비율, Steam 비추천 맥락 기준에서 출시 전 우선 점검이 필요한 이슈로 분류함. 적용 규칙: 상: 반... |

## prelaunch_condition_issue_summary.csv

- 설명: 조건값별 이슈 반복 요약 테이블

| 컬럼명 | 설명 | 예시 값 |
| --- | --- | --- |
| `condition_type` | 분석 조건의 종류 | genre |
| `condition_value` | 분석 조건의 실제 값 | Action |
| `issue_name_kor` | LLM 이슈 태그를 한국어 분석명으로 정리한 값 | 긍정 칭찬 |
| `issue_game_count` | 해당 이슈가 1회 이상 등장한 게임 수 | 57 |
| `positive_game_count` | 해당 이슈가 Steam 추천 리뷰 맥락에서 등장한 게임 수 | 56 |
| `negative_game_count` | 해당 이슈가 Steam 비추천 리뷰 맥락에서 등장한 게임 수 | 14 |
| `tag_positive_game_count` | LLM이 해당 이슈를 긍정 태그로 분류한 게임 수 | 57 |
| `tag_negative_game_count` | LLM이 해당 이슈를 부정 태그로 분류한 게임 수 | 0 |
| `tag_mixed_game_count` | LLM이 해당 이슈를 혼합 태그로 분류한 게임 수 | 0 |
| `high_urgency_game_count` | 해당 이슈가 High urgency 리뷰에서 등장한 게임 수 | 13 |
| `both_positive_negative_game_count` | 해당 이슈가 추천/비추천 맥락 모두에서 등장한 게임 수 | 13 |
| `total_issue_review_count` | 해당 이슈가 등장한 전체 리뷰 수 | 915 |
| `condition_game_count` | 해당 조건에 포함된 게임 수 | 62 |
| `issue_game_ratio` | 조건 내 게임 중 해당 이슈가 등장한 게임 비율 | 91.9 |
| `priority_level` | 반복성 및 Steam 비추천 맥락 기준으로 계산한 규칙 기반 점검 중요도. 개선 우선순위가 아니라 출시 전 확인 우선도에 가까움 | 상 |
| `priority_rule_detail` | 우선순위 산정에 적용된 규칙 설명 | 상: 반복 게임 수와 Steam 비추천 맥락이 모두 기준 이상 |
| `priority_reason` | 우선순위가 부여된 이유를 문장으로 정리한 값 | 여러 게임 반복성, 조건 내 발생 비율, Steam 비추천 맥락 기준에서 출시 전 우선 점검이 필요한 이슈로 분류함. 적용 규칙: 상: 반... |

## 출력 테이블 설명

`prelaunch_game_base.csv`
- 설명: 게임 1개 단위 출시 전 LLM 분석 요약 테이블

| 컬럼명                   | 설명                                               | 예시 값                                 |
| --------------------- | ------------------------------------------------ | ------------------------------------ |
| `appid`               | Steam 게임 고유 ID                                   | 571740                               |
| `game_name`           | Steam 게임명                                        | Golf It!                             |
| `genres_text`         | 게임의 Steam 장르 목록을 쉼표로 연결한 값                       | Casual, Indie, Simulation, Sports    |
| `price_group`         | 게임 가격을 분석용 구간으로 나눈 값                             | 0-5                                  |
| `top_steam_tags_text` | 게임의 주요 Steam 태그 목록을 쉼표로 연결한 값                    | Multiplayer, Mini Golf, Golf, Casual |
| `categories_text`     | Steam 카테고리 목록을 쉼표로 연결한 값                         | Single-player, Multi-player, PvP     |
| `play_style`          | 카테고리 정보를 바탕으로 분류한 플레이 방식                         | 멀티/협동 요소 포함                          |
| `review_count`        | 해당 게임에서 LLM 분석에 사용된 리뷰 수                         | 48                                   |
| `llm_positive_count`  | LLM이 긍정으로 분류한 리뷰 수                               | 27                                   |
| `llm_negative_count`  | LLM이 부정으로 분류한 리뷰 수                               | 15                                   |
| `llm_mixed_count`     | LLM이 혼합으로 분류한 리뷰 수                               | 4                                    |
| `high_urgency_count`  | LLM이 High urgency로 분류한 리뷰 수                      | 9                                    |
| `llm_positive_ratio`  | LLM 긍정 리뷰 비율                                     | 56.2                                 |
| `llm_negative_ratio`  | LLM 부정 리뷰 비율                                     | 31.2                                 |
| `high_urgency_ratio`  | LLM High urgency 리뷰 비율. 우선순위 계산 기준이 아니라 보조 참고 지표 | 18.8                                 |


`prelaunch_issue_repeat_summary.csv`
- 설명: 전체 게임 기준 이슈 반복성 요약 테이블

| 컬럼명                                 | 설명                                          | 예시 값                                       |
| ----------------------------------- | ------------------------------------------- | ------------------------------------------ |
| `issue_name_kor`                    | LLM 이슈 태그를 한국어 분석명으로 정리한 값                  | 긍정 칭찬                                      |
| `issue_game_count`                  | 해당 이슈가 1회 이상 등장한 게임 수                       | 140                                        |
| `positive_game_count`               | 해당 이슈가 Steam 추천 리뷰 맥락에서 등장한 게임 수            | 139                                        |
| `negative_game_count`               | 해당 이슈가 Steam 비추천 리뷰 맥락에서 등장한 게임 수           | 28                                         |
| `tag_positive_game_count`           | LLM이 해당 이슈를 긍정 태그로 분류한 게임 수                 | 140                                        |
| `tag_negative_game_count`           | LLM이 해당 이슈를 부정 태그로 분류한 게임 수                 | 0                                          |
| `tag_mixed_game_count`              | LLM이 해당 이슈를 혼합 태그로 분류한 게임 수                 | 0                                          |
| `high_urgency_game_count`           | 해당 이슈가 High urgency 리뷰에서 등장한 게임 수. 보조 참고 지표 | 31                                         |
| `both_positive_negative_game_count` | 해당 이슈가 추천/비추천 맥락 모두에서 등장한 게임 수              | 27                                         |
| `total_issue_review_count`          | 해당 이슈가 등장한 전체 리뷰 수                          | 2719                                       |
| `total_game_count`                  | 분석 대상 전체 게임 수                               | 151                                        |
| `issue_game_ratio`                  | 전체 게임 중 해당 이슈가 등장한 게임 비율                    | 92.7                                       |
| `priority_level`                    | 반복성 및 Steam 비추천 맥락 기준으로 계산한 규칙 기반 점검 중요도    | 상                                          |
| `priority_rule_detail`              | 우선순위 산정에 적용된 규칙 설명                          | 상: 반복 게임 수와 Steam 비추천 맥락이 모두 기준 이상         |
| `priority_reason`                   | 우선순위가 부여된 이유를 문장으로 정리한 값                    | 여러 게임 반복성과 Steam 비추천 맥락 기준에서 출시 전 우선 점검 필요 |




`prelaunch_condition_issue_summary.csv`
- 설명: 장르·가격대·Steam 태그·플레이 방식 조건별 이슈 반복 요약 테이블

| 컬럼명                                 | 설명                                          | 예시 값                                     |
| ----------------------------------- | ------------------------------------------- | ---------------------------------------- |
| `condition_type`                    | 분석 조건의 종류                                   | genre                                    |
| `condition_value`                   | 분석 조건의 실제 값                                 | Action                                   |
| `issue_name_kor`                    | LLM 이슈 태그를 한국어 분석명으로 정리한 값                  | 긍정 칭찬                                    |
| `issue_game_count`                  | 해당 조건에서 해당 이슈가 1회 이상 등장한 게임 수               | 57                                       |
| `positive_game_count`               | 해당 이슈가 Steam 추천 리뷰 맥락에서 등장한 게임 수            | 56                                       |
| `negative_game_count`               | 해당 이슈가 Steam 비추천 리뷰 맥락에서 등장한 게임 수           | 14                                       |
| `tag_positive_game_count`           | LLM이 해당 이슈를 긍정 태그로 분류한 게임 수                 | 57                                       |
| `tag_negative_game_count`           | LLM이 해당 이슈를 부정 태그로 분류한 게임 수                 | 0                                        |
| `tag_mixed_game_count`              | LLM이 해당 이슈를 혼합 태그로 분류한 게임 수                 | 0                                        |
| `high_urgency_game_count`           | 해당 이슈가 High urgency 리뷰에서 등장한 게임 수. 보조 참고 지표 | 13                                       |
| `both_positive_negative_game_count` | 해당 이슈가 추천/비추천 맥락 모두에서 등장한 게임 수              | 13                                       |
| `total_issue_review_count`          | 해당 이슈가 등장한 전체 리뷰 수                          | 915                                      |
| `condition_game_count`              | 해당 조건에 포함된 게임 수                             | 62                                       |
| `issue_game_ratio`                  | 조건 내 게임 중 해당 이슈가 등장한 게임 비율                  | 91.9                                     |
| `priority_level`                    | 반복성 및 Steam 비추천 맥락 기준으로 계산한 규칙 기반 점검 중요도    | 상                                        |
| `priority_rule_detail`              | 우선순위 산정에 적용된 규칙 설명                          | 상: 반복 게임 수와 Steam 비추천 맥락이 모두 기준 이상       |
| `priority_reason`                   | 우선순위가 부여된 이유를 문장으로 정리한 값                    | 여러 게임 반복성과 조건 내 발생 비율 기준에서 출시 전 우선 점검 필요 |
| `issue_direction`                   | 이슈를 강화 요소, 리스크 요소, 참고 요소 중 하나로 해석한 값        | 강화 요소                                    |





`prelaunch_checklist_evidence_base.csv`
- 설명: 출시 전 체크리스트 생성을 위해 LLM 프롬프트에 넣는 조건별 근거 테이블

| 컬럼명                                 | 설명                                          | 예시 값                                                        |
| ----------------------------------- | ------------------------------------------- | ----------------------------------------------------------- |
| `condition_type`                    | 분석 조건의 종류                                   | genre                                                       |
| `condition_value`                   | 분석 조건의 실제 값                                 | Action                                                      |
| `issue_name_kor`                    | LLM 이슈 태그를 한국어 분석명으로 정리한 값                  | 긍정 칭찬                                                       |
| `issue_direction`                   | 이슈 해석 방향                                    | 강화 요소                                                       |
| `condition_game_count`              | 해당 조건에 포함된 게임 수                             | 62                                                          |
| `issue_game_count`                  | 해당 이슈가 1회 이상 등장한 게임 수                       | 57                                                          |
| `issue_game_ratio`                  | 조건 내 게임 중 해당 이슈가 등장한 게임 비율                  | 91.9                                                        |
| `positive_game_count`               | 해당 이슈가 Steam 추천 리뷰 맥락에서 등장한 게임 수            | 56                                                          |
| `negative_game_count`               | 해당 이슈가 Steam 비추천 리뷰 맥락에서 등장한 게임 수           | 14                                                          |
| `tag_positive_game_count`           | LLM이 해당 이슈를 긍정 태그로 분류한 게임 수                 | 57                                                          |
| `tag_negative_game_count`           | LLM이 해당 이슈를 부정 태그로 분류한 게임 수                 | 0                                                           |
| `tag_mixed_game_count`              | LLM이 해당 이슈를 혼합 태그로 분류한 게임 수                 | 0                                                           |
| `high_urgency_game_count`           | 해당 이슈가 High urgency 리뷰에서 등장한 게임 수. 보조 참고 지표 | 13                                                          |
| `both_positive_negative_game_count` | 해당 이슈가 추천/비추천 맥락 모두에서 등장한 게임 수              | 13                                                          |
| `total_issue_review_count`          | 해당 이슈가 등장한 전체 리뷰 수                          | 915                                                         |
| `priority_level`                    | 규칙 기반 출시 전 점검 중요도                           | 상                                                           |
| `priority_rule_detail`              | 우선순위 산정에 적용된 규칙 설명                          | 상: 반복 게임 수와 Steam 비추천 맥락이 모두 기준 이상                          |
| `priority_reason`                   | 우선순위가 부여된 이유를 문장으로 정리한 값                    | 여러 게임 반복성과 Steam 비추천 맥락 기준에서 출시 전 우선 점검 필요                  |
| `llm_evidence_text`                 | 체크리스트 생성을 위해 LLM에 제공할 근거 문장                 | genre 조건 'Action'에서 '긍정 칭찬' 이슈는 전체 62개 게임 중 57개 게임에서 반복되었다. |



## tableau_allgames_prelaunch_source.csv

- 설명: Tableau 대시보드용 통합 원천 테이블

| 컬럼명 | 설명 | 예시 값 |
| --- | --- | --- |
| `appid` | Steam 게임 고유 ID | 571740 |
| `game_name` | Steam 게임명 | Golf It! |
| `condition_type` | 분석 조건의 종류 | price_group |
| `condition_type_kor` | 분석 조건 종류의 한글명 | 가격대 |
| `condition_value` | 분석 조건의 실제 값 | 0-5 |
| `genres_text` | 게임의 Steam 장르 목록을 쉼표로 연결한 값 | Casual, Indie, Simulation, Sports |
| `price_group` | 게임 가격을 분석용 구간으로 나눈 값 | 0-5 |
| `top_steam_tags_text` | 게임의 주요 Steam 태그 목록을 쉼표로 연결한 값 | Multiplayer, Mini Golf, Golf, Casual, Sports, Funny, Online Co-Op, Singleplay... |
| `categories_text` | Steam 카테고리 목록을 쉼표로 연결한 값 | Single-player, Multi-player, PvP, Online PvP, Shared/Split Screen PvP, Cross-... |
| `play_style` | 카테고리 정보를 바탕으로 분류한 플레이 방식 | 멀티/협동 요소 포함 |
| `issue_name_kor` | LLM 이슈 태그를 한국어 분석명으로 정리한 값 | UI/UX |
| `issue_review_count` | 해당 게임에서 해당 이슈가 등장한 리뷰 수 | 4 |
| `positive_review_count` | 해당 게임에서 해당 이슈가 Steam 추천 리뷰에 등장한 수 | 2 |
| `negative_review_count` | 해당 게임에서 해당 이슈가 Steam 비추천 리뷰에 등장한 수 | 2 |
| `mixed_review_count` | 해당 게임에서 해당 이슈가 혼합 리뷰에 등장한 수 | 0 |
| `high_urgency_review_count` | 해당 게임에서 해당 이슈가 High urgency 리뷰에 등장한 수 | 2 |
| `tag_positive_present` | 해당 게임에서 이 이슈의 긍정 태그가 존재하는지 여부 | False |
| `tag_negative_present` | 해당 게임에서 이 이슈의 부정 태그가 존재하는지 여부 | True |
| `tag_mixed_present` | 해당 게임에서 이 이슈의 혼합 태그가 존재하는지 여부 | False |
| `high_urgency_present` | 해당 게임에서 이 이슈가 High urgency 리뷰에 등장했는지 여부 | True |
| `both_positive_negative_present` | 해당 게임에서 이 이슈가 추천/비추천 맥락 모두에 등장했는지 여부 | True |
| `review_count` | 해당 게임에서 LLM 분석에 사용된 리뷰 수 | 48 |
| `llm_positive_count` | LLM이 긍정으로 분류한 리뷰 수 | 27 |
| `llm_negative_count` | LLM이 부정으로 분류한 리뷰 수 | 15 |
| `llm_mixed_count` | LLM이 혼합으로 분류한 리뷰 수 | 4 |
| `high_urgency_count` | LLM이 High urgency로 분류한 리뷰 수 | 9 |
| `llm_positive_ratio` | LLM 긍정 리뷰 비율 | 56.2 |
| `llm_negative_ratio` | LLM 부정 리뷰 비율 | 31.2 |
| `high_urgency_ratio` | LLM High urgency 리뷰 비율 | 18.8 |
| `condition_game_count` | 해당 조건에 포함된 게임 수 | 54 |
| `issue_game_count` | 해당 이슈가 1회 이상 등장한 게임 수 | 24 |
| `issue_game_ratio` | 조건 내 게임 중 해당 이슈가 등장한 게임 비율 | 44.4 |
| `positive_game_count` | 해당 이슈가 Steam 추천 리뷰 맥락에서 등장한 게임 수 | 18 |
| `negative_game_count` | 해당 이슈가 Steam 비추천 리뷰 맥락에서 등장한 게임 수 | 12 |
| `tag_positive_game_count` | LLM이 해당 이슈를 긍정 태그로 분류한 게임 수 | 6 |
| `tag_negative_game_count` | LLM이 해당 이슈를 부정 태그로 분류한 게임 수 | 19 |
| `tag_mixed_game_count` | LLM이 해당 이슈를 혼합 태그로 분류한 게임 수 | 0 |
| `high_urgency_game_count` | 해당 이슈가 High urgency 리뷰에서 등장한 게임 수 | 11 |
| `both_positive_negative_game_count` | 해당 이슈가 추천/비추천 맥락 모두에서 등장한 게임 수 | 6 |
| `priority_level` | 반복성 및 Steam 비추천 맥락 기준으로 계산한 규칙 기반 점검 중요도. 개선 우선순위가 아니라 출시 전 확인 우선도에 가까움 | 상 |
| `priority_rule_detail` | 우선순위 산정에 적용된 규칙 설명 | 상: 반복 게임 수와 Steam 비추천 맥락이 모두 기준 이상 |
| `priority_order` | 대시보드 정렬용 우선순위 숫자 | 1 |
| `tag_dna_game_count` | 해당 조건에서 태그 DNA 기준에 포함된 게임 수 | 918 |
| `top_performance_game_count` | 해당 조건에서 상위 성과 기준에 포함된 게임 수 | 356 |
| `top_performance_ratio` | 해당 조건에서 상위 성과 게임이 차지하는 비율 | 38.8 |
| `avg_positive_rate` | 해당 조건에 포함된 게임들의 평균 긍정률 | 81.4 |
| `median_total_reviews` | 해당 조건에 포함된 게임들의 전체 리뷰 수 중앙값 | 78 |